# SigTekX + AWS SageMaker: Anomaly Detection in Ionospheric Signals

This notebook demonstrates the ML pipeline integration of SigTekX with AWS SageMaker
Random Cut Forest (RCF) for detecting anomalies in VLF/ULF ionospheric signals.

**Pipeline:**
1. SigTekX produces spectral magnitude time-series (STFT)
2. Synthetic ionospheric disturbances are injected (solar flares, geomagnetic storms)
3. RCF (or local IsolationForest) detects anomalous spectral patterns
4. Results are visualized with detection metrics

**For class presentation:** Run cells top-to-bottom. The `--local-only` flag
uses sklearn (no AWS credentials needed for demo).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import anomaly detection functions
from scripts.aws.anomaly_detection import (
    generate_spectral_timeseries,
    inject_anomalies,
    run_local_detection,
    compute_detection_metrics,
)

print('Imports OK')

## 1. Generate Synthetic Spectral Data

Simulates SigTekX STFT output at 48kHz sample rate with 4096-point FFT —
matching our ionosphere research configuration.

In [ ]:
# Generate 5 minutes of synthetic spectral data
times, normal_data, frame_rate = generate_spectral_timeseries(
    duration_sec=300.0,
    sample_rate_hz=48000.0,
    nfft=4096,
    overlap=0.75,
    n_freq_bins=64,
)

print(f'Generated {normal_data.shape[0]} frames x {normal_data.shape[1]} frequency bins')
print(f'Frame rate: {frame_rate:.1f} Hz')
print(f'Duration: {times[-1]:.1f} seconds')

In [ ]:
# Visualize the normal spectral data
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    normal_data.T, aspect='auto', origin='lower',
    extent=[times[0], times[-1], 0, normal_data.shape[1]],
    cmap='viridis'
)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency Bin')
ax.set_title('Normal Ionospheric Spectral Pattern (Simulated SigTekX Output)')
plt.colorbar(im, label='Magnitude')
plt.tight_layout()
plt.show()

## 2. Inject Ionospheric Disturbances

Three types of anomalies simulating real ionospheric phenomena:
- **Power spike**: Sudden broadband increase (solar flare / Sudden Ionospheric Disturbance)
- **Frequency shift**: Enhanced low-frequency power (geomagnetic storm onset)
- **Narrowband burst**: Single-frequency spike (VLF transmitter interference)

In [ ]:
# Split data: 60% train (normal), 40% test (with injected anomalies)
split_idx = int(len(normal_data) * 0.6)
train_data = normal_data[:split_idx]
test_base = normal_data[split_idx:]
test_times = times[split_idx:]

# Inject anomalies into test set
test_data, anomaly_meta = inject_anomalies(
    test_base, test_times, frame_rate, n_anomalies=5
)

# Adjust frame indices relative to test set
for a in anomaly_meta:
    a['start_frame'] -= split_idx
    a['end_frame'] -= split_idx

print(f'Training set: {train_data.shape[0]} frames (normal)')
print(f'Test set: {test_data.shape[0]} frames (with anomalies)')
print(f'\nInjected anomalies:')
for a in anomaly_meta:
    print(f'  {a["type"]:20s} at t={a["start_time"]:.1f}s - {a["end_time"]:.1f}s')

In [ ]:
# Visualize test data with anomalies
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Normal (baseline)
ax1.imshow(
    test_base.T, aspect='auto', origin='lower',
    extent=[test_times[0], test_times[-1], 0, test_base.shape[1]],
    cmap='viridis'
)
ax1.set_ylabel('Freq Bin')
ax1.set_title('Baseline (Normal)')

# With anomalies
ax2.imshow(
    test_data.T, aspect='auto', origin='lower',
    extent=[test_times[0], test_times[-1], 0, test_data.shape[1]],
    cmap='viridis'
)
for a in anomaly_meta:
    ax2.axvspan(a['start_time'], a['end_time'], alpha=0.3, color='red')
ax2.set_ylabel('Freq Bin')
ax2.set_xlabel('Time (s)')
ax2.set_title('With Injected Disturbances (red regions)')

plt.tight_layout()
plt.show()

## 3. Anomaly Detection

Using **IsolationForest** (sklearn) as a local stand-in for SageMaker RCF.
Both are tree-based unsupervised anomaly detectors — conceptually identical.

In production, replace with `run_sagemaker_rcf()` for the AWS-native version.

In [ ]:
# Train model on normal data, score test data
scores = run_local_detection(train_data, test_data)

print(f'Scored {len(scores)} frames')
print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]')
print(f'Mean score: {scores.mean():.4f}')

## 4. Results & Metrics

In [ ]:
# Compute detection metrics
metrics = compute_detection_metrics(scores, anomaly_meta, len(test_data))

print('Detection Metrics:')
print(f'  Precision: {metrics["precision"]:.3f}')
print(f'  Recall:    {metrics["recall"]:.3f}')
print(f'  F1 Score:  {metrics["f1"]:.3f}')
print(f'  Threshold: {metrics["threshold"]:.4f}')
print(f'  Detected:  {metrics["n_detected"]}/{len(scores)} frames')

In [ ]:
# Full visualization
threshold = metrics['threshold']
detected = scores > threshold

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Spectrogram with anomaly overlay
ax1 = axes[0]
im = ax1.imshow(
    test_data.T, aspect='auto', origin='lower',
    extent=[test_times[0], test_times[-1], 0, test_data.shape[1]],
    cmap='viridis'
)
for a in anomaly_meta:
    ax1.axvspan(a['start_time'], a['end_time'], alpha=0.3, color='red')
ax1.set_ylabel('Frequency Bin')
ax1.set_title('SigTekX Spectral Output with Injected Disturbances')
plt.colorbar(im, ax=ax1, label='Magnitude')

# Panel 2: Anomaly scores
ax2 = axes[1]
ax2.plot(test_times[:len(scores)], scores, color='darkred', linewidth=0.5)
ax2.axhline(y=threshold, color='orange', linestyle='--',
            label=f'Threshold ({threshold:.3f})')
for a in anomaly_meta:
    ax2.axvspan(a['start_time'], a['end_time'], alpha=0.2, color='red')
ax2.set_ylabel('Anomaly Score')
ax2.set_title('Anomaly Detection Scores')
ax2.legend()

# Panel 3: Detection result
ax3 = axes[2]
ax3.fill_between(test_times[:len(scores)], detected.astype(float),
                 alpha=0.5, color='red', label='Detected')
for a in anomaly_meta:
    ax3.axvspan(a['start_time'], a['end_time'], alpha=0.2, color='blue',
               label='Ground Truth')
ax3.set_ylabel('Anomaly Detected')
ax3.set_xlabel('Time (seconds)')
ax3.set_title(f'Detection Results (Precision={metrics["precision"]:.2f}, Recall={metrics["recall"]:.2f}, F1={metrics["f1"]:.2f})')
ax3.set_ylim(-0.1, 1.5)
# Deduplicate legend entries
handles, labels = ax3.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax3.legend(by_label.values(), by_label.keys())

plt.tight_layout()
plt.show()

## 5. Summary

**Key takeaways:**
- SigTekX spectral output feeds directly into ML anomaly detection pipelines
- Random Cut Forest / IsolationForest effectively detects ionospheric disturbances
- The pipeline scales to cloud GPU instances via SageMaker Processing Jobs
- Real-time monitoring possible by streaming SigTekX output to a deployed RCF endpoint

**Production deployment:**
```python
# Replace run_local_detection() with:
from scripts.aws.anomaly_detection import run_sagemaker_rcf
scores = run_sagemaker_rcf(train_data, test_data, role_arn)
```